## nb_03_game_plays_silver

Cleans and validates bronze table and writes the result to a silver table
Every cleaning/validation rule lives in its own function (defined once,
below) and is then applied one step at a time in its own cell, so each
intermediate result can be inspected before moving to the next step.

### Imports

In [18]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    col, count, when, lit, min, max,
    substring, concat, to_timestamp, to_date,
)

StatementMeta(, 4a9c8038-930b-4447-ae52-492f5f63fc34, 38, Finished, Available, Finished, False)

### Load common db functions

In [19]:
%run nb_00_dbutils

StatementMeta(, 4a9c8038-930b-4447-ae52-492f5f63fc34, 48, Finished, Available, Finished, True)

### Parameters

`RUN_PIPELINE` controls whether the "Run the pipeline" steps below actually execute.
It defaults to `True` for normal, standalone runs of this notebook.

When this notebook is loaded from another notebook via `%run` (e.g. from a test
notebook), pass `RUN_PIPELINE = False` as a run parameter so only the function/config
definitions are loaded and the pipeline against the real `bronze.tablename` / `silver.tablename`
tables is skipped:

```
%run <notebook name> { "RUN_PIPELINE": false }
```

In [20]:
# This cell is tagged "parameters" so Fabric/Synapse can override it when the
# notebook is invoked with %run nb_02_game_silver { "RUN_PIPELINE": false }
RUN_PIPELINE: bool = True

StatementMeta(, 4a9c8038-930b-4447-ae52-492f5f63fc34, 49, Finished, Available, Finished, False)

### Config

In [21]:
BRONZE_TABLE = "bronze.game_plays"
SILVER_TABLE = "silver.game_plays"

# Only these columns make it into the silver table
required_cols: list[str] = [
    "play_id",
    "game_id",
    "event",
]

# Natural key used to de-duplicate rows
DEDUPE_KEYS: list[str] = ["play_id"]
PRIMARY_KEYS: list[str] = ["play_id"]

# Prevents table from loading when called from tests
if RUN_PIPELINE:

    # Set up foreign keys for integrity checks
    GAME_FOREIGN_KEYS: list[str] = ["game_id"]
    game: DataFrame = load_table(spark, "bronze.game", GAME_FOREIGN_KEYS)


StatementMeta(, 4a9c8038-930b-4447-ae52-492f5f63fc34, 50, Finished, Available, Finished, False)

✅ Loaded bronze.game: 26305 rows


## Run the pipeline
Each step runs in its own cell so the result can be inspected before moving on.

In [22]:
if RUN_PIPELINE:
    df = load_table(spark, BRONZE_TABLE, columns=required_cols)

    df = remove_duplicates(df, columns=DEDUPE_KEYS)
    df = validate_no_nulls(df, columns=required_cols)

    df = validate_foreign_keys(df, ftable=game, keys=GAME_FOREIGN_KEYS)

    # Validate primary key(s) to ensure integrity
    df = validate_primary_keys(df, keys=PRIMARY_KEYS)

    write_table(df, SILVER_TABLE)
    print("🏁 Silver load complete.")

StatementMeta(, 4a9c8038-930b-4447-ae52-492f5f63fc34, 51, Finished, Available, Finished, False)

✅ Loaded bronze.game_plays: 5050529 rows
🔁 Removed 833466 duplicate row(s) based on ['play_id']
✅ Primary key check passed — ['play_id'] is unique across 4217063 row(s)
✅ Foreign key check passed — game_id
✅ Wrote silver.game_plays (4217063 rows, 3 columns)
🏁 Silver load complete.
